In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/EEG_TSSL_DATA"

windows = np.load(os.path.join(DATA_PATH, "windows.npy"))
labels = np.load(os.path.join(DATA_PATH, "labels_windows.npy"))
trial_ids = np.load(os.path.join(DATA_PATH, "trial_ids.npy"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from sklearn.model_selection import train_test_split

unique_trials = np.unique(trial_ids)
trial_labels = np.array([labels[trial_ids == t][0] for t in unique_trials])

train_trials, _ = train_test_split(
    unique_trials,
    test_size=0.3,
    random_state=42,
    stratify=trial_labels
)

train_mask = np.isin(trial_ids, train_trials)
X_train = windows[train_mask]

print("Train windows:", X_train.shape)


Train windows: (54656, 32, 256)


In [ ]:
# ================== NEW MULTI-TASK DATASET (T-S-F-G) ==================
class EEGMultiTaskSSLDataset(Dataset):
    def __init__(self, X, trial_ids):
        self.X = X
        self.trial_ids = trial_ids
        self.stimulus_id = trial_ids % 40  # Same video = same stimulus (for inter-subject contrast)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx].copy()
        x = torch.from_numpy(x).float()

        # 1. Temporal SSL (your original)
        v1 = resize(crop(x))
        v2 = resize(crop(x))

        # 2. Spatial Jigsaw
        x_spatial = spatial_jigsaw(x)

        # 3. Frequency Jigsaw (we use same x, but predict permutation label in loss)
        x_freq = x.clone()  # placeholder - label handled in loop

        # 4. Transformation Recognition
        x_trans, trans_label = apply_random_transform(x)

        # 5. Inter-subject positive pair (same stimulus, different subject)
        same_stim = np.where(self.stimulus_id == self.stimulus_id[idx])[0]
        same_stim = same_stim[same_stim != idx]
        if len(same_stim) > 0:
            pos_idx = np.random.choice(same_stim)
            x_pos = torch.from_numpy(self.X[pos_idx]).float()
            v_pos = resize(crop(x_pos))
        else:
            v_pos = v1.clone()  # fallback

        return v1, v2, x_spatial, x_freq, x_trans, trans_label, v_pos

In [ ]:
# Crop & Resize (your original)
def crop(sig, ratio=0.9):
    C, T = sig.shape
    L = int(T * ratio)
    s = np.random.randint(0, T - L)
    return sig[:, s:s+L]

def resize(sig, target=256):
    return F.interpolate(sig.unsqueeze(0), size=target, mode="linear", align_corners=False).squeeze(0)

# Spatial Jigsaw (GMSS style)
def spatial_jigsaw(x):
    x = x.clone() # x is (32, 256)

    # Identify channels not defined in region_groups
    all_ch_indices = set(range(x.shape[0])) # all 32 channels (0 to 31)
    grouped_channels_flat = set(ch for group in region_groups for ch in group) # the 30 channels defined in region_groups
    ungrouped_channels = sorted(list(all_ch_indices - grouped_channels_flat)) # [29, 30]

    # Create 'blocks' to permute: existing region groups + individual ungrouped channels
    # This ensures all 32 channels are accounted for and permuted
    blocks_to_permute = list(region_groups) + [[ch] for ch in ungrouped_channels]

    random.shuffle(blocks_to_permute)

    new_order = []
    for block in blocks_to_permute:
        new_order.extend(block)

    x = x[new_order, :]
    return x

# Signal Transformations (Wu et al.)
def apply_random_transform(x):
    transforms = [identity, add_noise, magnitude_warping, time_permutation, cropping]
    idx = random.randint(0, 4)
    return transforms[idx](x), idx

def identity(x): return x
def add_noise(x, sigma=0.05): return x + sigma * torch.randn_like(x)
def magnitude_warping(x, sigma=0.2):
    return x * (1 + sigma * torch.randn(x.shape[0], 1, device=x.device))
def time_permutation(x, n_segments=5):
    original_length = x.shape[-1]
    seg_len = original_length // n_segments
    segments = []
    for i in range(n_segments):
        start = i * seg_len
        # Ensure the last segment covers any remaining length
        end = (i + 1) * seg_len if i < n_segments - 1 else original_length
        segments.append(x[..., start:end])

    random.shuffle(segments)
    permuted_x = torch.cat(segments, dim=-1)

    # As a safeguard, interpolate if the length is still not the original
    if permuted_x.shape[-1] != original_length:
        permuted_x = F.interpolate(permuted_x.unsqueeze(1), size=original_length, mode='linear', align_corners=False).squeeze(1)

    return permuted_x
def cropping(x, ratio=0.9):
    L = int(x.shape[-1] * ratio)
    start = random.randint(0, x.shape[-1] - L)
    cropped = x[..., start:start+L]
    return F.interpolate(cropped.unsqueeze(1), size=x.shape[-1], mode='linear').squeeze(1)

# Frequency views (extend to 5 bands)
def get_frequency_views(x):
    return {
        "delta": bandpass(x, 1, 4),
        "theta": bandpass(x, 4, 8),
        "alpha": bandpass(x, 8, 13),
        "beta":  bandpass(x, 13, 30),
        "gamma": bandpass(x, 30, 45)
    }

In [ ]:
ssl_loader = DataLoader(
    EEGMultiTaskSSLDataset(X_train, trial_ids[train_mask]),
    batch_size=16,
    shuffle=True
)

In [ ]:
class CNNTransformerEncoder(nn.Module):
    def __init__(self, emb_dim=128, n_heads=4, n_layers=2):
        super().__init__()

        # CNN front-end (same spirit as before)
        self.conv1 = nn.Conv1d(32, 64, kernel_size=7, padding=3)
        self.conv2 = nn.Conv1d(64, emb_dim, kernel_size=5, padding=2)
        self.pool = nn.MaxPool1d(4)   # 256 → 64

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=n_heads,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )

    def forward(self, x):
        # x: (B, 32, 256)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)              # (B, 128, 64)

        x = x.permute(0, 2, 1)        # (B, 64, 128)
        x = self.transformer(x)       # (B, 64, 128)

        x = x.mean(dim=1)             # temporal pooling
        return x                      # (B, 128)


In [ ]:
# ================== REQUIRED FOR NEW ENCODER ==================
import random

# Adjacency matrix for GraphConv (32 channels)
adj = torch.eye(32, device=device)
for i in range(32):
    for j in range(max(0, i-4), min(32, i+5)):
        if i != j:
            adj[i, j] = 1.0
adj = adj / adj.sum(dim=1, keepdim=True)

# Brain region groups for Spatial Jigsaw (from GMSS paper)
region_groups = [
    [0,1,27,26],   # Pre-frontal
    [2,28,24,23],  # Frontal
    [3,4,5],       # Left frontal
    [22,25],       # Right frontal
    [6,7,20,21],   # Central/Temporal
    [8,9,18,19],   # Parietal
    [10,11,16,17], # Left/Right parietal
    [12,13,15],    # Occipital
    [14,31]        # OZ/O2
]

In [ ]:
class GraphConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.adj = adj
        self.linear = nn.Linear(in_channels, out_channels)

    def forward(self, x):
        x = torch.matmul(self.adj, x)          # Graph convolution on channels
        x = x.permute(0, 2, 1)                 # (B, T, C)
        x = self.linear(x)
        x = x.permute(0, 2, 1)                 # (B, C, T)
        return x

class CNNTransformerEncoder(nn.Module):
    def __init__(self, emb_dim=128, n_heads=4, n_layers=2):
        super().__init__()

        # NEW: GraphConv front-end
        self.graph_conv = GraphConv(32, 64)

        # Your original CNN + Transformer
        self.conv1 = nn.Conv1d(64, 64, kernel_size=7, padding=3)
        self.conv2 = nn.Conv1d(64, emb_dim, kernel_size=5, padding=2)
        self.pool = nn.MaxPool1d(4)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim, nhead=n_heads, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # NEW: Multi-task heads for Jigsaw + Transform
        self.spatial_jigsaw_head = nn.Linear(emb_dim, len(region_groups))
        self.freq_jigsaw_head     = nn.Linear(emb_dim, 5)      # 5 frequency bands
        self.transform_head       = nn.Linear(emb_dim, 6)      # 5 transforms + identity

    def forward(self, x):
        x = self.graph_conv(x)                 # NEW
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)                       # (B, 128, 64)

        x = x.permute(0, 2, 1)                 # (B, 64, 128)
        x = self.transformer(x)
        x = x.mean(dim=1)                      # (B, 128)
        return x

In [ ]:
# ===== Load encoder from Notebook-2 (transfer learning) =====
encoder = CNNTransformerEncoder().to(device)   # Creates NEW architecture

load_path = "/content/drive/MyDrive/EEG_TSSL_DATA/encoder_cnn_transformer_tfssl.pt"
checkpoint_state_dict = torch.load(load_path, map_location=device)

# Filter out incompatible keys from the checkpoint state_dict.
# The 'conv1' layer in the new model expects 64 input channels due to GraphConv,
# but the old checkpoint's 'conv1' was for 32 input channels.
# GraphConv and the new multi-task heads are also new and should be randomly initialized.
filtered_state_dict = {
    k: v for k, v in checkpoint_state_dict.items()
    if not k.startswith('conv1.') # Exclude conv1 weights due to size mismatch
}

encoder.load_state_dict(
    filtered_state_dict,
    strict=False          # Allows other new layers/missing keys to be ignored
)

encoder.to(device)
print(f"Loaded compatible weights from {load_path} (strict=False, with filtering for conv1)")
print("New GraphConv, conv1, and multi-task heads are randomly initialized")

Loaded compatible weights from /content/drive/MyDrive/EEG_TSSL_DATA/encoder_cnn_transformer_tfssl.pt (strict=False, with filtering for conv1)
New GraphConv, conv1, and multi-task heads are randomly initialized


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = CNNTransformerEncoder().to(device)

optimizer = torch.optim.Adam(
    encoder.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

encoder.train()


CNNTransformerEncoder(
  (graph_conv): GraphConv(
    (linear): Linear(in_features=32, out_features=64, bias=True)
  )
  (conv1): Conv1d(64, 64, kernel_size=(7,), stride=(1,), padding=(3,))
  (conv2): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
  (pool): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (d

In [ ]:
class SpatialProjector(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(256, 64)

    def forward(self, x):
        # x: (B, C, T)
        B, C, T = x.shape
        x = x.view(B * C, T)
        z = self.fc(x)
        return z.view(B, C, -1)   # (B, C, 64)


In [ ]:
spatial_proj = SpatialProjector().to(device)


In [ ]:
def raw_corr(x):
    # x: (C, T)
    x = x - x.mean(dim=1, keepdim=True)
    corr = torch.matmul(x, x.T) / (x.shape[1] - 1)
    return corr / torch.norm(corr)


In [ ]:
def spatial_loss(x, proj):
    B = x.size(0)
    loss = 0.0

    for i in range(B):
        eeg = x[i]                    # (C, T)
        corr_raw = raw_corr(eeg)

        z = proj(eeg.unsqueeze(0)).squeeze(0)   # (C, 64)
        corr_enc = torch.matmul(z, z.T) / z.size(1)
        corr_enc = corr_enc / torch.norm(corr_enc)

        loss += F.mse_loss(corr_enc, corr_raw)

    return loss / B


In [ ]:
def nt_xent(z1, z2, temp=0.5):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    B = z1.size(0)
    reps = torch.cat([z1, z2], dim=0)
    sim = torch.matmul(reps, reps.T)

    mask = torch.eye(2 * B, device=z1.device).bool()
    sim = sim[~mask].view(2 * B, -1)

    pos = torch.cat([torch.sum(z1 * z2, dim=1),
                     torch.sum(z2 * z1, dim=1)])

    logits = torch.cat([pos.unsqueeze(1), sim], dim=1)
    labels = torch.zeros(2 * B, dtype=torch.long).to(z1.device)

    return F.cross_entropy(logits / temp, labels)


In [ ]:
lambda_spatial = 0.05   # keep small
optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(spatial_proj.parameters()),
    lr=1e-4
)


In [ ]:
from scipy.signal import butter, filtfilt

def bandpass(x, low, high, fs=128, order=4):
    b, a = butter(order, [low/(fs/2), high/(fs/2)], btype='band')
    return filtfilt(b, a, x, axis=-1)

def get_frequency_views(x):
    return {
        "alpha": bandpass(x, 8, 13),
        "beta":  bandpass(x, 13, 30)
    }


In [ ]:
# ================== NEW MULTI-TASK TRAINING LOOP ==================
EPOCHS = 30

for epoch in range(EPOCHS):
    total_loss = 0.0
    for v1, v2, x_spatial, x_freq, x_trans, trans_label, v_pos in ssl_loader:
        v1 = v1.to(device)
        v2 = v2.to(device)
        x_spatial = x_spatial.to(device)
        x_trans = x_trans.to(device)
        trans_label = trans_label.to(device)
        v_pos = v_pos.to(device)

        z1 = encoder(v1)
        z2 = encoder(v2)
        z_sp = encoder(x_spatial)
        z_tr = encoder(x_trans)
        z_pos = encoder(v_pos)

        loss_t = nt_xent(z1, z2)
        loss_t_inter = nt_xent(z1, z_pos) * 0.5   # inter-subject contrast
        loss_s = spatial_loss(x_spatial, spatial_proj)
        loss_f = 1 - torch.mean(torch.sum(F.normalize(encoder(x_freq.to(device)), dim=1) * F.normalize(z1, dim=1), dim=1))
        loss_trans = F.cross_entropy(encoder.transform_head(z_tr), trans_label)
        loss_spatial_jigsaw = F.cross_entropy(encoder.spatial_jigsaw_head(z_sp), torch.zeros(z_sp.size(0), dtype=torch.long, device=device))  # dummy for now - can be improved

        loss = (1.0 * loss_t + 0.5 * loss_t_inter + 0.3 * loss_s + 0.3 * loss_f + 0.2 * loss_trans + 0.1 * loss_spatial_jigsaw)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | T-S-F-G Jigsaw+Transform Loss: {total_loss:.4f}")

Epoch 1/30 | T-S-F-G Jigsaw+Transform Loss: 10896.4264
Epoch 2/30 | T-S-F-G Jigsaw+Transform Loss: 10391.3244
Epoch 3/30 | T-S-F-G Jigsaw+Transform Loss: 10150.4157
Epoch 4/30 | T-S-F-G Jigsaw+Transform Loss: 10055.0518
Epoch 5/30 | T-S-F-G Jigsaw+Transform Loss: 10005.2091
Epoch 6/30 | T-S-F-G Jigsaw+Transform Loss: 9961.9584
Epoch 7/30 | T-S-F-G Jigsaw+Transform Loss: 9861.5389
Epoch 8/30 | T-S-F-G Jigsaw+Transform Loss: 9704.3894
Epoch 9/30 | T-S-F-G Jigsaw+Transform Loss: 9658.1396
Epoch 10/30 | T-S-F-G Jigsaw+Transform Loss: 9626.0709
Epoch 11/30 | T-S-F-G Jigsaw+Transform Loss: 9602.5361
Epoch 12/30 | T-S-F-G Jigsaw+Transform Loss: 9588.5054
Epoch 13/30 | T-S-F-G Jigsaw+Transform Loss: 9573.0950
Epoch 14/30 | T-S-F-G Jigsaw+Transform Loss: 9564.4260
Epoch 15/30 | T-S-F-G Jigsaw+Transform Loss: 9555.0266
Epoch 16/30 | T-S-F-G Jigsaw+Transform Loss: 9547.0388
Epoch 17/30 | T-S-F-G Jigsaw+Transform Loss: 9533.3432
Epoch 18/30 | T-S-F-G Jigsaw+Transform Loss: 9528.8926
Epoch 19/30 | 

In [ ]:
save_path = "/content/drive/MyDrive/EEG_TSSL_DATA/encoder_tsf_g_jigsaw.pt"
encoder.eval()
torch.save(encoder.state_dict(), save_path)
print(f"[Notebook-4] Final T-S-F-G encoder saved to: {save_path}")

[Notebook-4] Final T-S-F-G encoder saved to: /content/drive/MyDrive/EEG_TSSL_DATA/encoder_tsf_g_jigsaw.pt
